# Mapping Electric Scooter & Bike Availability Against Commuting Patterns

An ArcPy pipeline that summarizes shared scooter and bike availability by street segment, runs Hot Spot (Getis-Ord Gi*) analysis to find where availability clusters, then brings in ACS commuting-mode data for Hennepin County so those hot spots can be compared against how residents already get to work. The core pipeline is also wrapped into a reusable function so it can be re-run against a different year's availability data without repeating every step by hand.

## Setup

In [2]:
import arcpy
import os

# update this to your own project geodatabase
arcpy.env.workspace = r"C:\path\to\your\project\ScooterAvailability.gdb"

## Step 1: Converting street centerlines to points

The join and Hot Spot steps below both need point features, not lines, so the street centerline layer is converted to centroid points first.

In [18]:
centerline_input = "PW_Street_Centerline"
centerline_output = "Street_centroids_1"

arcpy.management.FeatureToPoint(in_features=centerline_input, out_feature_class=centerline_output, point_location="CENTROID")
print("centerline conversion complete")

centerline conversion complete


## Step 2: Summarizing scooter/bike availability by street segment

Availability records are grouped by `ClosestCenterlineID` and summarized to the mean and sum of `NumberAvailable` per segment.

In [25]:
in_table = "Scooter_and_Bike_Availability_2024"
out_table = "Scooter_Bike_Summary_1"

statistics_fields = [["NumberAvailable", "MEAN"], ["NumberAvailable", "SUM"]]
case_field = ["ClosestCenterlineID"]

arcpy.analysis.Statistics(in_table=in_table, out_table=out_table, statistics_fields=statistics_fields, case_field=case_field)
print("success!")

success!


## Step 3: Preparing both tables to join

The summary table's `ClosestCenterlineID` field includes decimal points, while the centroid table's `GBSID` field doesn't - so the two need reconciling before they can be joined. Centerline IDs belonging to trail segments (rather than streets) also don't have a match in the centroid table and are removed first. (The resulting field was spot-checked in the table view before proceeding.)

In [28]:
# non-numeric centerline IDs in the summary table are trail segments, which have no match
# in the street centroid table, so they're removed before joining
feature_class = "Scooter_Bike_Summary_1"
field_name = "ClosestCenterlineID"
output_layer_name = "Updated_Scooter_Bike_Summary_1"
expression = f"{field_name} NOT LIKE '%00'"

arcpy.management.MakeTableView(feature_class, output_layer_name)
arcpy.management.SelectLayerByAttribute(in_layer_or_view=output_layer_name, selection_type="NEW SELECTION", where_clause=expression)
arcpy.management.DeleteRows(output_layer_name)
print("rows that contain trail information have been deleted")

rows that contain trail information have been deleted


In [29]:
# add a text field to hold a reformatted, joinable version of GBSID
input_table = "Street_Centroids_1"
new_field_name = "NewCenterline"

arcpy.management.AddField(in_table=input_table, field_name=new_field_name, field_type="TEXT")
print("success!")

success!


In [30]:
# populate the new field with GBSID + ".00" so it matches the summary table's ID format
feature_class = "Street_Centroids_1"
new_field = "NewCenterline"
expression = "str(!GBSID!) + '.00'"

arcpy.management.CalculateField(in_table=feature_class, field=new_field, expression=expression)
print("feature has been added")

feature has been added


## Step 4: Joining the tables

In [31]:
end_features = "Street_Centroids_1"
end_join_field = "NewCenterline"
join_features = "Updated_Scooter_Bike_Summary_1"
join_field = "ClosestCenterlineID"
added_fields = ["MEAN_NumberAvailable", "SUM_NumberAvailable"]

arcpy.management.JoinField(in_data=end_features, in_field=end_join_field, join_table=join_features, join_field=join_field, fields=added_fields)
print("successfully joined")

successfully joined


## Step 5: Hot Spot analysis on 2024 availability

Getis-Ord Gi* is run on both the sum and mean of available scooters/bikes per segment, using a fixed distance band with Manhattan distance - chosen since it suited the centroid features better than the default Euclidean option.

In [32]:
arcpy.stats.HotSpots(Input_Feature_Class="Street_Centroids_1", Input_Field="SUM_NumberAvailable", Output_Feature_Class="HotSpot_SUM",
                      Conceptualization_of_Spatial_Relationships="FIXED_DISTANCE_BAND", Distance_Method="MANHATTAN_DISTANCE")

<Result 'ScooterAvailability.gdb\\HotSpot_SUM'>

In [33]:
arcpy.stats.HotSpots(Input_Feature_Class="Street_Centroids_1", Input_Field="MEAN_NumberAvailable", Output_Feature_Class="HotSpot_MEAN",
                      Conceptualization_of_Spatial_Relationships="FIXED_DISTANCE_BAND", Distance_Method="MANHATTAN_DISTANCE")

<Result 'ScooterAvailability.gdb\\HotSpot_MEAN'>

## Wrapping the pipeline into a reusable function

Steps 1 through 5 above are packaged into a single function, so the same process can be re-run against a different year's dataset without repeating every step by hand. Applied here to 2020 availability data.

In [35]:

def scooter_bike_processing(scooter_input_dataset):
    ##convert centerlines to centroids (pt. 1)
    centerline_input = "PW_Street_Centerline"
    centerline_output = "Street_Centroids"
    arcpy.management.FeatureToPoint(in_features = centerline_input, out_feature_class = centerline_output, point_location = "CENTROID")
    ##summary stats (pt.2)
    in_table = scooter_input_dataset
    out_table = f"{scooter_input_dataset}_Summary"
    statistics_fields = [["NumberAvailable", "MEAN"], ["NumberAvailable", "SUM"]]
    case_field = ["ClosestCenterlineID"]
    arcpy.analysis.Statistics(in_table=in_table, out_table=out_table, statistics_fields=statistics_fields, case_field=case_field)
    ##clean datasets to prepare for joining (pt.3)
    ##this will delete trail datasets
    feature_class = f"{scooter_input_dataset}_Summary"
    field_name = "ClosestCenterlineID"
    output_layer_name = f"Updated_{scooter_input_dataset}_Summary"
    expression = f"{field_name} NOT LIKE '%00'"
    arcpy.management.MakeTableView(feature_class, output_layer_name)
    arcpy.management.SelectLayerByAttribute (in_layer_or_view=output_layer_name, selection_type="NEW SELECTION", where_clause=expression)
    arcpy.management.DeleteRows(output_layer_name)
    ##joining process
    ##add field to centerline set to add .00 
    input_table = "Street_Centroids"
    source_field = "GBSID"
    new_field_name = "NewCenterline"
    arcpy.management.AddField(in_table=input_table, field_name=new_field_name, field_type="TEXT")
    ##now actually add the .00
    feature_class = "Street_Centroids"
    source_field = "GBSID"
    new_field ="NewCenterline"
    expression = "str(!GBSID!) + '.00'"
    arcpy.management.CalculateField(in_table=feature_class, field=new_field, expression=expression)
    ##Now we actually join
    end_features = "Street_Centroids"
    end_join_field = "NewCenterline"
    join_features = f"Updated_{scooter_input_dataset}_Summary"
    join_field = "ClosestCenterlineID"
    added_fields = ["MEAN_NumberAvailable", "SUM_NumberAvailable"]
    arcpy.management.JoinField(in_data=end_features, in_field=end_join_field, join_table=join_features, join_field=join_field, fields=added_fields)
    ##Hotspot analysis 
    arcpy.stats.HotSpots (Input_Feature_Class="Street_Centroids", Input_Field="SUM_NumberAvailable", Output_Feature_Class="HotSpot_SUM",
                      Conceptualization_of_Spatial_Relationships = "FIXED_DISTANCE_BAND", Distance_Method = "MANHATTAN_DISTANCE")
    arcpy.stats.HotSpots (Input_Feature_Class="Street_Centroids", Input_Field="MEAN_NumberAvailable", Output_Feature_Class="HotSpot_MEAN",
                      Conceptualization_of_Spatial_Relationships = "FIXED_DISTANCE_BAND", Distance_Method = "MANHATTAN_DISTANCE")

scooter_input_dataset = "Scooter_Availability_2020"
scooter_bike_processing(scooter_input_dataset)

## Bringing in ACS commuting data for comparison

To see whether scooter/bike availability lines up with how people already commute, 5-year ACS commuting-mode data (table DP03) is pulled in for Hennepin County and run through the same Hot Spot analysis, so the results can be compared against the scooter/bike hot spots above.

In [36]:
import csv

# update these to point at your own downloaded ACS DP03 5-year table and project geodatabase
csv_file = r"C:\path\to\your\ACSDP5Y2023.DP03-Data.csv"
output_gdb = r"C:\path\to\your\project\ScooterAvailability.gdb"
output_table_name = "MN_Census_2023"
output_table = os.path.join(output_gdb, output_table_name)

arcpy.conversion.TableToTable(csv_file, output_gdb, output_table_name)

<Result 'ScooterAvailability.gdb\\MN_Census_2023'>

Only the commuting-mode columns of interest are pulled out into a separate CSV, to keep the join simple.

In [50]:
arcpy.env.workspace = output_gdb
input_data = "MN_Census_2023"

output_filename = "MN_Commmutes_2023.csv"
output_file_path = os.path.join(arcpy.env.workspace, output_filename)

columns = ["OBJECTID", "GEO_ID", "NAME", "DP03_0018PE", "DP03_0019PE", "DP03_0020PE", "DP03_0021PE", "DP03_0022PE", "DP03_0023PE", "DP03_0024PE"]

with open(output_file_path, 'w', newline='', encoding='utf-8') as output_csv:
    csv_writer = csv.writer(output_csv)
    csv_writer.writerow(columns)
    with arcpy.da.SearchCursor(input_data, columns) as cursor:
        for row in cursor:
            csv_writer.writerow(row)

## Limiting the census tracts to Hennepin County

In [51]:
feature_layer = "tl_2023_27_tract"
field = "GEOID"
output_feature_class = "Hennepin_county"

# GEOID values starting with 27053 are Hennepin County, Minnesota
sql_expression = f"{field} LIKE '27053%'"

arcpy.management.SelectLayerByAttribute(in_layer_or_view=feature_layer, selection_type="NEW_SELECTION", where_clause=sql_expression)
arcpy.management.CopyFeatures(feature_layer, output_feature_class)

<Result 'ScooterAvailability.gdb\\Hennepin_county'>

## Joining commuting data to the tracts

In [58]:
end_features = "tl_2023_27_tract"
end_join_field = "GEOIDFQ"
join_features = "MN_Commmutes_2023.csv"
join_field = "GEO_ID"
added_fields = ["OBJECTID", "GEO_ID", "NAME", "DP03_0018PE", "DP03_0019PE", "DP03_0020PE", "DP03_0021PE", "DP03_0022PE", "DP03_0023PE", "DP03_0024PE"]
output_features = "Joined_table"

arcpy.management.JoinField(in_data=end_features, in_field=end_join_field, join_table=join_features, join_field=join_field, fields=added_fields)

<Result 'tl_2023_27_tract'>

## Hot Spot analysis on commuting patterns

For each commuting mode, a percent field is added to the Hennepin County tracts and run through the same Getis-Ord Gi* analysis used on the scooter/bike data - this time with queen contiguity (tracts share edges and corners) and Euclidean distance, the more standard choice for polygon data.

### Commuting by other means

In [43]:
input_table = "Hennepin_county"
og_field = "DP03_0023P"
new_field = "Other_Means_Percent"

arcpy.management.AddField(input_table, new_field, "DOUBLE")
arcpy.management.CalculateField(input_table, new_field, f"!{og_field}!", "PYTHON3")

arcpy.stats.HotSpots(Input_Feature_Class="Hennepin_county", Input_Field="Other_Means_Percent", Output_Feature_Class="HotSpot_Other_Means",
                      Conceptualization_of_Spatial_Relationships="CONTIGUITY_EDGES_CORNERS", Distance_Method="EUCLIDEAN_DISTANCE")

<Result 'Hennepin_county'>

<Result 'ScooterAvailability.gdb\\HotSpot_Other_Means'>

### Commuting by walking

In [44]:
input_table = "Hennepin_county"
og_field = "DP03_0022P"
new_field = "Walked_Percent"

arcpy.management.AddField(input_table, new_field, "DOUBLE")
arcpy.management.CalculateField(input_table, new_field, f"!{og_field}!", "PYTHON3")

arcpy.stats.HotSpots(Input_Feature_Class="Hennepin_county", Input_Field="Walked_Percent", Output_Feature_Class="HotSpot_walked",
                      Conceptualization_of_Spatial_Relationships="CONTIGUITY_EDGES_CORNERS", Distance_Method="EUCLIDEAN_DISTANCE")

<Result 'Hennepin_county'>

<Result 'ScooterAvailability.gdb\\HotSpot_walked'>

### Commuting by public transit

In [45]:
input_table = "Hennepin_county"
og_field = "DP03_0021P"
new_field = "Publictransit_Percent"

arcpy.management.AddField(input_table, new_field, "DOUBLE")
arcpy.management.CalculateField(input_table, new_field, f"!{og_field}!", "PYTHON3")

arcpy.stats.HotSpots(Input_Feature_Class="Hennepin_county", Input_Field="Publictransit_Percent", Output_Feature_Class="HotSpot_publictransit",
                      Conceptualization_of_Spatial_Relationships="CONTIGUITY_EDGES_CORNERS", Distance_Method="EUCLIDEAN_DISTANCE")

<Result 'Hennepin_county'>

<Result 'ScooterAvailability.gdb\\HotSpot_publictransit'>

### Commuting overall

In [57]:
input_table = "Hennepin_county"
og_field = "DP03_0018P"
new_field = "Percent_commuting"

arcpy.management.AddField(input_table, new_field, "DOUBLE")
arcpy.management.CalculateField(input_table, new_field, f"!{og_field}!", "PYTHON3")

arcpy.stats.HotSpots(Input_Feature_Class="Hennepin_county", Input_Field="Percent_commuting", Output_Feature_Class="HotSpot_commuting",
                      Conceptualization_of_Spatial_Relationships="CONTIGUITY_EDGES_CORNERS", Distance_Method="EUCLIDEAN_DISTANCE")

<Result 'Hennepin_county'>

<Result 'ScooterAvailability.gdb\\HotSpot_commuting'>